# FORESEE Model: Inelastic Dark Matter 

## Load Libraries 

In [ ]:
import sys, os
src_path = "../../"
sys.path.append(src_path)
import numpy as np
from src.foresee import Foresee, Utility, Model
from matplotlib import pyplot as plt
from IDMCalc import InelasticDarkMatter

In [ ]:
#Create symlink to direct production spectra. obtain_direct_production()
#writes the decayed chi_2 spectra to files/direct/iDM/, which the model reads
#through model/direct/ (the A' tables under files/direct/DarkPhoton/ are read
#by obtain_direct_production directly, not through this symlink).
try: os.unlink('model/direct')
except: pass
os.symlink(
    src=os.path.normpath('../../../files/direct/iDM'),
    dst='model/direct',
    target_is_directory=True,
)

## 1. Specifying the Model

Here we consider the inelastic dark matter (iDM) model introduced in [1810.01879](https://arxiv.org/pdf/1810.01879.pdf). The models consists of a dark photon ($A'$) with mass $m_{A'}$, and two nearly degenerate states in the dark sector ($\chi_1,~\chi_2$), with $m_2>m_1$ and $\Delta=\frac{m_2-m_1}{m_1}$. The dark photon - dark sector interaction has an off-diagonal coupling and the dark photon with coupling strength  $\alpha_D = \epsilon_D^2/(4\pi)$. The corresppning Lagrangian is,


\begin{equation}
\mathcal{L} = - \frac{1}{2} \textcolor{red}{m_1} \bar{\chi_1} \chi_1 - \frac{1}{2} \textcolor{red}{m_1} \bar{\chi_1} -\chi_1 - \left(i \textcolor{red}{\epsilon_D} A'_{\mu}\bar{\chi_2}\gamma^{\mu}\chi_1 + h.c\right) - \frac{1}{2} \textcolor{red}{m_{A'}}^2 A'^2  - \textcolor{red}{\epsilon} e \sum \bar f \gamma^\mu f A'_\mu.
\end{equation}

The free parameters of the model are usually chosen to be $m_{A'}$, $r=\frac{m_{A'}}{m_1}$, $\Delta=\frac{m_2-m_1}{m_1}$, $\alpha_D$ and $\epsilon$. We are interested in the case where $alpha_D$ is sufficiently large and $m_1+m_2 < m_{A'}$ such that BR($A'\rightarrow \chi_1 \chi_2) \approx 1$. In practice, we will make some choice for $r$, $\Delta$, $\alpha_D$ and then plot the reach in $\epsilon$ vs $m_{2}$. 

**FORESEE assumes that the mass variable is the mass of the LLP, for example to calculate $\gamma = E/m$. That's why we chose m2 as mass variable.** 

In [ ]:
energy = "14"
modelname="iDM"
model = Model(modelname, path="./")

**Model Initialization:** The production and decay rates of the iDM depend not only on the choice of $m_{A'}$ and $\epsilon$, but also $\alpha_D$, $r$ and $\Delta$. To simplify the process of setting up the model we provide the class `InelasticDarkMatter(alphaD, delta, r)`.

In [ ]:
# Builder parameters: canonical build.py / load_model() defaults.
nsample_3body = 2000
generators_light = ["EPOSLHC", "SIBYLL", "QGSJET"][:1]
brem_configurations = ["Brem_QRA_L1.5", "Brem_QRA_L1.0", "Brem_QRA_L2.0"][:1]
alphaD, delta, r = 0.1, 0.1, 3

In [ ]:
idm = InelasticDarkMatter(alphaD=alphaD, delta=delta, r=r)

 The function `obtain_ctau_br()` will calculate the lifetime and branching fraction. An approximate expression for decay width of $X_2$ is given by Eq. 13 in [1810.01879](https://arxiv.org/pdf/1810.01879.pdf): $ \Gamma (\chi_2 \rightarrow \chi_1 l^+ l^-) = \frac{4\epsilon^2\alpha_{EM}\alpha_{D}}{15\pi}\frac{\Delta^5m_1^5}{m^4_{A'}}$. 

In [ ]:
# obtain_ctau_br() recomputes the chi_2 lifetime for the chosen
# (alphaD, delta, r) and writes the resulting table to model/ctau.txt.
idm.obtain_ctau_br()

Some of the most relevant production models, Brem and DY production, are loaded through the `foresee.add_production_direct()`. This requires that we have already obtained the spectra of the LLP and saved under `model/direct/...`. 

We can make use of the known dark photon spectra, which are saved in `files/direct/DarkPhoton` and obtain the $X_2$ spectra by decaying $A' \to X_1 X_2$ for the given benchmark masses. This is done by `obtain_direct_production()`. 


In [ ]:
idm.obtain_direct_production(energy=energy, nsample=10)

**Production:** If the dark photon is sufficiently light, it is primarily produced in the decay of pseudoscalar mesons $\pi^0$, $\eta$ and $\eta' \to \gamma A'$. The branching fractions for the leading channels are 

\begin{equation}
    \text{BR}(\pi^0 \to A' \gamma) = 2 \epsilon^2 \times\text{BR}(\pi^0 \to \gamma\gamma) \times \left(1-m_{A'}^2/m_\pi^2\right)^3
    \quad\quad\text{and}\quad\quad
    \text{BR}(\eta \to A' \gamma) = 2 \epsilon^2 \times\text{BR}(\eta \to \gamma\gamma) \times \left(1-m_{A'}^2/m_\eta^2\right)^3
\end{equation}

We can use `model.add_production_3bodydecay()` function with `integration = "chain_decay"` to model this. This requires the input `br = [branching ratio function, mass of the intermediate state]`, where we choose $m_{\phi} = 2$GeV as an example. We also specify the `massrange` and set `scaling = 2`: the $\chi_2$ yield scales as $\epsilon^2$ (the $A'$ is produced through kinetic mixing, $\propto \epsilon^2$, then $A' \to \chi_1 \chi_2$ with BR $\approx 1$). 

In the following, we model the production using `EPOSLHC`, `SIBYLL`, and `QGSJET`.

In [ ]:
massap = "mass*"+str(r)+"/(1+"+str(delta)+")"

model.add_production_3bodydecay(
    pid0 = "111",
    pid1 = "22",
    pid2 = "0",
    integration = "chain_decay",
    br = ["2.*0.99*coupling**2*pow(1.-pow("+massap+"/self.masses(111),2),3)", massap],
    generator = generators_light,
    energy = energy,
    nsample = nsample_3body,
    massrange = [0,model.masses(111)*(1+delta)/r]
)

model.add_production_3bodydecay(
    pid0 = "221",
    pid1 = "22",
    pid2 = "0",
    integration = "chain_decay",
    br = ["2.*0.39*coupling**2*pow(1.-pow("+massap+"/self.masses(221),2),3)", massap],
    generator = generators_light,
    energy = energy,
    nsample = nsample_3body,
    massrange = [0,model.masses(221)*(1+delta)/r]
)

model.add_production_3bodydecay(
    pid0 = "331",
    pid1 = "22",
    pid2 = "0",
    integration = "chain_decay",
    br = ["2.*0.023*coupling**2*pow(1.-pow("+massap+"/self.masses(331),2),3)", massap],
    generator = generators_light,
    energy = energy,
    nsample = nsample_3body,
    massrange = [0,model.masses(331)*(1+delta)/r]
)

Dark photons can also be produced via dark Bremsstrahlung and DY production, which we load through `model.add_production_direct()`. 

In [ ]:
model.add_production_direct(
    label = "Brem",
    configuration = brem_configurations,
    energy = energy,
    coupling_ref=1,
    masses = idm.get_brem_masses(),
)

In [ ]:
model.add_production_direct(
    label = "DY",
    energy = energy,
    coupling_ref=1,
    masses = idm.get_dy_masses(),
    condition='True',
)

**Decay:** The decay length (and later also the branching fraction) that we previously generated. 

In [ ]:
model.set_ctau_1d(
    filename="model/ctau.txt", 
)

We can now initiate FORESEE with the model that we just created. 

In [ ]:
foresee = Foresee(path=src_path)
foresee.set_model(model=model)

## 2. Event Generation

In the following, we want to study one specific benchmark point with $m_{2}=1$ GeV and $\epsilon= 10^{-3}$. 

In [ ]:
mass, coupling, = 1.0, 1e-3

First, we will produce the corresponding flux for this mass and a reference coupling $\epsilon_{ref}=1$. 

In [ ]:
plot=foresee.get_llp_spectrum(mass=mass, coupling=1, do_plot=True)
os.makedirs(f"figures/{modelname}", exist_ok=True)
plot.savefig(f"figures/{modelname}/Spectrum_{modelname}.pdf", bbox_inches="tight")
plot.show()

Next, let us define the configuration of the detector (in terms of position, size and luminosity). Here we choose FASER.

In [ ]:
foresee.set_detector(
    distance=650, 
    selection="np.sqrt((x.x)**2 + (x.y)**2)<1",    
    length=10, 
    luminosity=3000, 
)

For our benchmark point, let us now look at how many particle decay inside the decay volume. We also export 1000 unweighted events as a HEPMC file. 

In [ ]:
setupnames  = generators_light

momenta, weights, _ = foresee.write_events(
    mass = mass, 
    coupling = coupling, 
    energy = energy, 
    numberevent = 1000,
    filename = "model/events/test.hepmc", 
    return_data = True,
    weightnames=setupnames,
    modes=None,
)

for isetup, setup in enumerate(setupnames):
    print("Expected number of events for "+setup+":", round(sum(weights[:,isetup]),3))

Let us plot the resulting energy distribution

In [ ]:
fig = plt.figure(figsize=(7,5))
ax = plt.subplot(1,1,1)
energies = [p.e for p in momenta], 
for isetup, setup in enumerate(setupnames):
    ax.hist(energies, weights=weights[:,isetup], bins=np.logspace(2,4, 20+1), histtype='step', label=setup) 
ax.set_xscale("log")
ax.set_xlim(1e2,1e4) 
ax.set_xlabel("E [GeV]") 
ax.set_ylabel("Number of Events per Bin") 
ax.legend(frameon=False, labelspacing=0, fontsize=14, loc='upper left')
os.makedirs(f"figures/{modelname}", exist_ok=True)
plt.savefig(f"figures/{modelname}/E_distribution_{modelname}.pdf", bbox_inches="tight")
plt.show()

## Sensitivity Reach

In the following, we will obtain the projected sensitivity for the LLP model. For this, we first define a grid of couplings and masses, and then produce the corresponding fluxes. 

In [ ]:
masses = [round(x,5) for x in np.logspace(-2,1,50)] + idm.get_brem_masses() + idm.get_dy_masses()
thresholds = [
    0.13093, 0.13498, 0.13903, 0.53143, 0.54786, 0.5643, 0.92905, 0.95778,
    0.98651,
]
masses = sorted(masses + thresholds)
couplings = np.logspace(-7,0,100)


# Use cached LLP spectra: get_llp_spectrum recomputes on every call, 
# so skip any masses already saved in model/LLP_spectra/.
for mass in masses:
    if not os.path.exists(f"model/LLP_spectra/{energy}TeV_m_{mass}.txt.gz"):
        foresee.get_llp_spectrum(mass=mass, coupling=1)

In [ ]:
%%time

productions = [
    {"channels": ["111"] , "color": "red"      , "label": r"$\pi^0 \to \gamma \chi_2\bar{\chi_2}$", "generators": generators_light},
    {"channels": ["221"] , "color": "orange"   , "label": r"$\eta \to \gamma \chi_2\bar{\chi_2}$" , "generators": generators_light},
    {"channels": ["331"] , "color": "blue"   , "label": r"$\eta' \to \gamma \chi_2\bar{\chi_2}$" , "generators": generators_light},
    {"channels": ["Brem"], "color": "limegreen", "label": r"Bremsstrahlung"       , "generators": brem_configurations},
    {"channels": ["DY"]  , "color": "tab:purple", "label": r"Drell-Yan"            , "generators": ["DY"]},
]

branchings = []

plot=foresee.plot_production(
    masses = masses,
    productions = productions,
    energy=energy,
    condition = "logth<-3.7 and logp>2",
    xlims=[0.01,10],ylims=[1e1,1e13],
    xlabel=r"Mass [GeV]",
    ylabel=r"Production Rate $\sigma/\epsilon^2$ [pb]",
    title=r"$\theta < 0.2$ mrad and $E > 100$ GeV",
    legendloc=(0.97,1),
    fs_label=12,
    ncol=2,
    figsize=(7,6),
)

os.makedirs(f"figures/{modelname}", exist_ok=True)
plot.savefig(f"figures/{modelname}/Production_{modelname}.pdf", bbox_inches="tight")
plot.show()

Let us now scan over various masses and couplings, and record the resulting number of evets. Note that here we again consider the FASER configuration, which we set up before.

In [ ]:
setupnames = ['EPOSLHC']
modes = None

if energy == "13.6": detectors = [["FASER_R3"  , 480, "np.sqrt(x.x**2 + x.y**2)< .1", 1.5, 250 ,  None]]
elif energy == "14": detectors = [["FASER_HL"  , 480, "np.sqrt(x.x**2 + x.y**2)< .1", 1.5, 3000,  None], 
                                  ["FASER2_HL" , 650, "-1.5<x.x<1.5 and -.5<x.y<.5" , 10 , 3000,  None]]

condition = f"np.sqrt(p**2 + mass**2) > 100"

for detector in detectors: 

    #setup detector
    dlabel, distance, selection, length, luminosity, channels  = detector

    #skip detectors already precomputed (the plot cell reads them); scan only missing ones
    if all(os.path.exists(f"model/results/{energy}TeV_{dlabel}_{label}.npy") for label in setupnames):
        continue

    foresee.set_detector(distance=distance, selection=selection, length=length, luminosity=luminosity, channels=channels)

    #get reach  
    list_nevents = {label:[] for label in setupnames}
    for mass in masses:
        couplings, _, nevents, _, _  = foresee.get_events(mass=mass, energy=energy, couplings = couplings,modes=modes,nsample=10, preselectioncuts = condition)
        for i,label in enumerate(setupnames): list_nevents[label].append(nevents.T[i])  
            
    #save results
    configuration=dlabel
    for label in setupnames: 
        result = np.array([masses,couplings,list_nevents[label]], dtype='object')
        np.save("model/results/"+energy+"TeV_"+configuration+"_"+label+".npy",result)

We can now plot the results. For this, we first specify all detector setups for which we want to show result (filename in model/results directory, label, color, linestyle, opacity alpha for filled contours, required number of events).

In [ ]:
setups = [ 
    ["13.6TeV_FASER_R3_EPOSLHC.npy"   , r"FASER (Run 3)"    , "firebrick"         ,  "solid"  , 0., 3],
    ["14TeV_FASER_HL_EPOSLHC.npy"   , r"FASER (HL-LHC)"    , "red"         ,  "dashed"  , 0., 3],
    ["14TeV_FASER2_HL_EPOSLHC.npy"   , r"FASER2 (HL-LHC)"    , "salmon"         ,  "dashed"  , 0., 3],    
]

Then we specify all the existing bounds, separating the bounds obtained by experimental collaborations and theory recasts.

In [ ]:
bounds = [   
    ["bounds_E137.txt"       , "E137"          ,  1.2e-1, 1.3e-4, 45  ] ,       
    ["bounds_LSND.txt"       , "LSND"          ,  3e-2, 2e-5, 90  ] ,   
    ["bounds_BaBaR.txt"     , "BaBaR"        ,  5e-2, 1e-3, 0  ],
    ["bounds_LEP.txt"       , "LEP"          ,  1.5e-2, 2.6e-2, 0  ],
]

We then specify other projected sensitivitities (filename in model/bounds directory, color, label, label position x, label position y, label rotation)

In [ ]:
projections = [
    # ["limits_SeaQuest.txt"         , "orchid"       , "SeaQuest"         , 5e-2, 1.2e-5, 0  ],
    # ["limits_Belle2.txt"           , "purple"       , "Belle 2"          , 1e+0, 4.6e-5, 0  ],
    # ["limits_BaBar.txt"            , "magenta"      , "BaBar\n(DLJ)"     , 1.9e-1, 1.0e-2, -33  ],
    # ["limits_MiniBooNE.txt"        , "green"        , "MiniBooNE"        , 2.7e-2, 5e-3, 90  ],
    # ["limits_JSNS2.txt"            , "red"          , r"JSNS$^2$"        , 4.5e-2, 4e-2, -55  ],
    # ["limits_BDX.txt"              , "brown"        , "BDX"              , 1.5e-1, 4e-2, -60  ],
    # ["limits_LDMX.txt"             , "mediumvioletred", "LDMX"           , 3e-1, 3e-3, 90  ],
    # ["limits_LHCEWPT.txt"          , "darkolivegreen", "LHC (EWPT)"      , 3.2e0, 1.5e-2, 0  ],
    # ["limits_MATHUSLA.txt"         , "teal"         , "MATHUSLA"         , 5e0, 5.5e-5, 0  ],
    # ["limits_MATHUSLA2.txt"        , "teal"         , ""                 , 5e0, 5e-5, 0 ],
    # ["limits_Codex_b.txt"          , "turquoise"    , "Codex-B"          , 5e-2, 4e-3, 0  ],
    # ["limits_LHCDMJ.txt"           , "cyan"         , " LHC\n(DMJ)"      , 6.0e+1, 2e-3, 0  ],
    # ["limits_LHCb.txt"             , "blue"         , "LHCb"             , 4.5e+0, 2.5e-3, -33],
    # ["limits_LHCtiming.txt"        , "deepskyblue"  , "   LHC\n(timing)" , 5.0e+1, 8e-4, 0  ],
]

Finally, we can plot everything using `foresee.plot_reach()`.

In [ ]:
plot = foresee.plot_reach(
    setups=setups,
    bounds=bounds,
    bounds2=[],
    projections=projections,
    branchings=None,
    title="iDM", 
    xlims = [0.01,1e1], 
    ylims=[1e-5,1e-1],
    xlabel=r"Dark matter mass $m_{2}$ [GeV]", 
    ylabel=r"Kinetic Mixing $\epsilon$",
    legendloc=(0.9,0.2),
    linewidths=2,
)


os.makedirs(f"figures/{modelname}", exist_ok=True)
plot.savefig(f"figures/{modelname}/Reach_{modelname}.pdf", bbox_inches="tight")
plot.show()